In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

In [5]:
df = pd.read_json('results/judge_results.jsonl', lines=True)
df[:3]

,id,group,claim,pred_label,gold_label,correct,num_cited,num_verified,reasoning,cited,retried,first_pass
0,18079,SUPPORTS,Scandinavia includes the three kingdoms of Den...,SUPPORTS,SUPPORTS,True,2,2,Passage 1 states that the term Scandinavia is ...,[{'id': 'bd7d96d7-4689-5b65-8d8b-83eed1f1ff29'...,True,NOT ENOUGH INFO
1,198208,SUPPORTS,Saturn is smaller than Jupiter.,SUPPORTS,SUPPORTS,True,1,1,The evidence states that Saturn is 'the second...,[{'id': '8cbce921-9a9b-5d1d-b49c-5f6686c7910c'...,False,None
2,125557,SUPPORTS,Chile is a country.,SUPPORTS,SUPPORTS,True,1,1,The evidence explicitly states that Chile is a...,[{'id': '2cba9815-fe19-564a-9bf2-4c4e749b83ad'...,False,None


In [7]:
print(df.shape)
print(df["group"].value_counts())

(122, 12)
group
SUPPORTS    34
REFUTES     34
NEI-A       34
NEI-B       20
Name: count, dtype: int64


In [10]:
accuracy = df["correct"].mean()
print("Number of records:", len(df))
print("Number of Errors: ", (df["pred_label"] == "ERROR").sum())
print(f"Overall Accuracy: {accuracy:.2f}")

Number of records: 122
Number of Errors:  0
Overall Accuracy: 0.66


In [19]:
df["first_label"] = df["first_pass"].where(df["retried"], df["pred_label"])
print(f"First-pass Accuracy: {(df['first_label'] == df['gold_label']).mean():.2f}")

First-pass Accuracy: 0.69


In [11]:
accuracy_by_group = df.groupby("group")["correct"].agg(["mean", "count"])
print("\nAccuracy by Group:")
print(accuracy_by_group)


Accuracy by Group:
              mean  count
group                    
NEI-A     0.352941     34
NEI-B     0.500000     20
REFUTES   0.852941     34
SUPPORTS  0.852941     34


In [12]:
pd.crosstab(df["gold_label"], df["pred_label"])

pred_label,NOT ENOUGH INFO,REFUTES,SUPPORTS
gold_label,,,
NOT ENOUGH INFO,22,14,18
REFUTES,1,29,4
SUPPORTS,1,4,29


In [16]:
print(classification_report(df["gold_label"], df["pred_label"]))

                 precision    recall  f1-score   support

NOT ENOUGH INFO       0.92      0.41      0.56        54
        REFUTES       0.62      0.85      0.72        34
       SUPPORTS       0.57      0.85      0.68        34

       accuracy                           0.66       122
      macro avg       0.70      0.70      0.65       122
   weighted avg       0.74      0.66      0.64       122



In [20]:
print("Number of Citations verified:", df["num_verified"].sum() / df["num_cited"].sum())

Number of Citations verified: 0.9857142857142858


In [24]:
retried_df = df[df["retried"]]
first_okay = (retried_df["first_pass"] == retried_df["gold_label"])
final_okay = retried_df["correct"]

fixed = (final_okay & ~first_okay).sum()
print(f"Fixed {fixed}")

broke = (~final_okay & first_okay).sum()
print(f"Broke {broke}")

no_change =len(retried_df) - fixed - broke
print(f"No Change {no_change}")

Fixed 4
Broke 8
No Change 22


In [23]:
mistakes = df[~df["correct"]][["id", "group", "claim", "gold_label", "pred_label", "reasoning"]]
mistakes

,id,group,claim,gold_label,pred_label,reasoning
5,24448,SUPPORTS,Rupert Murdoch is the Chairman of News Corpora...,SUPPORTS,REFUTES,"The evidence states that in September 2023, Ru..."
7,63081,SUPPORTS,Rupert Murdoch has control of 21st Century Fox.,SUPPORTS,REFUTES,The evidence states that Rupert Murdoch 'was a...
11,2042,SUPPORTS,Pakistan is on the crossroads of Central and W...,SUPPORTS,REFUTES,The claim states that Pakistan is on the cross...
22,112565,SUPPORTS,Antigua and Barbuda was discovered in 1493.,SUPPORTS,REFUTES,The evidence states that hunter-gatherers sett...
29,142025,SUPPORTS,"South Island is referred to as the ""mainland"" ...",SUPPORTS,NOT ENOUGH INFO,Passage 1 explicitly mentions that 'The island...
36,61641,REFUTES,Francis I of France had the popular nickname F...,REFUTES,SUPPORTS,The evidence explicitly mentions that Francis ...
40,23148,REFUTES,Gin is a name.,REFUTES,SUPPORTS,"The claim states that 'Gin is a name', which i..."
42,54077,REFUTES,The Hundred Years' War includes the Civil War.,REFUTES,SUPPORTS,The evidence explicitly states that the Hundre...
44,35627,REFUTES,The Bahamas is unknown.,REFUTES,NOT ENOUGH INFO,The provided passages do not state that The Ba...
65,14855,REFUTES,"Rupert Murdoch is a broadcaster, forming BSkyB.",REFUTES,SUPPORTS,The evidence explicitly states that Murdoch fo...


In [ ]:
for _, r in mistakes.iterrows():
    print(f"ID: {r['id']}, Claim: {r['claim']}")
    print(f"Gold Label: {r['gold_label']}, Predicted Label: {r['pred_label']}")
    print(f"Reasoning: {r['reasoning']}\n")


ID: 24448, Claim: Rupert Murdoch is the Chairman of News Corporation.
Gold Label: SUPPORTS, Predicted Label: REFUTES
Reasoning: The evidence states that in September 2023, Rupert Murdoch announced he would be stepping down as chairman of Fox Corp. and News Corp. Therefore, the claim that he is the Chairman of News Corporation is refuted by the evidence of his resignation/stepping down.

ID: 63081, Claim: Rupert Murdoch has control of 21st Century Fox.
Gold Label: SUPPORTS, Predicted Label: REFUTES
Reasoning: The evidence states that Rupert Murdoch 'was also the owner of [...] 21st Century Fox (until 2019)'. Therefore, he does not currently have control of it.

ID: 2042, Claim: Pakistan is on the crossroads of Central and Western Asia.
Gold Label: SUPPORTS, Predicted Label: REFUTES
Reasoning: The claim states that Pakistan is on the crossroads of Central and Western Asia. Passage 1 states that Pakistan is 'Situated at the crossroads of South Asia, the Middle East, and Central Asia'. Sin

In [28]:
adjudication = {
    # fever_wrong
    24448:  ("fever_wrong", "outdated: Murdoch stepped down 2023"),
    63081:  ("fever_wrong", "outdated: 21st Century Fox until 2019"),
    61641:  ("fever_wrong", "evidence contains the nickname"),
    57109:  ("fever_wrong", "originally intended for adults, distinguished from YA"),
    72647:  ("fever_wrong", "article lists 163 floors"),
    223672: ("fever_wrong", "born in Bonn"),
    179323: ("fever_wrong", "started drawing in elementary school"),
    144498: ("fever_wrong", "politician, minister, party leader"),
    123439: ("fever_wrong", "evidence: UK Conservative Party, not US"),
    49323:  ("fever_wrong", "notes were printed"),
    101407: ("fever_wrong", "quote matches claim"),
    57559:  ("fever_wrong", "first computer at ~20, not 7"),
    24510:  ("fever_wrong", "ruled by Norway then Denmark"),
    25882:  ("fever_wrong", "Dubai + West Asia includes Arabian Peninsula"),
    134051: ("fever_wrong", "policies known as Thatcherism (borderline: laws vs policies)"),
    179334: ("fever_wrong", "lifespan dates imply young adulthood"),
    140934: ("fever_wrong", "Greek diaspora especially in Germany"),
    174968: ("fever_wrong", "feature film, not documentary"),
    168048: ("fever_wrong", "described with male pronouns"),
    127588: ("fever_wrong", "number one hip hop song 2008"),
    87098:  ("fever_wrong", "Prime Minister, not President"),
    184062: ("fever_wrong", "born in the Bronx"),
    96181:  ("fever_wrong", "commedia dell'arte character"),
    19921:  ("fever_wrong", "released 2012, not 2002"),

    # over_inference: REFUTES/SUPPORTS beyond the evidence
    54731:  ("over_inference", "says text does not state it, still REFUTES"),
    139938: ("over_inference", "common sense: human, not chimpanzee"),
    39708:  ("over_inference", "57 elevators; 'unused' not addressed"),
    107360: ("over_inference", "no mention of baseball -> REFUTES"),
    21729:  ("over_inference", "hip-hop dates from outside knowledge"),
    125163: ("over_inference", "no evidence -> REFUTES"),
    161043: ("over_inference", "vague 'in jeopardy' read into invasion"),

    # hearsay_error: Hearsay wrong
    2042:   ("hearsay_error", "too literal: Middle East = Western Asia"),
    112565: ("hearsay_error", "too literal on 'discovered'"),
    142025: ("hearsay_error", "missed support for 'cultured place'"),
    23148:  ("hearsay_error", "wordplay on 'the name gin'"),
    54077:  ("hearsay_error", "ambiguous 'the Civil War'"),
    35627:  ("hearsay_error", "too cautious: evidence contradicts 'unknown'"),
    14855:  ("hearsay_error", "Murdoch formed BSkyB but is not a broadcaster"),
    151360: ("hearsay_error", "1.6M accepted as 'almost 2 million'"),
    85256:  ("hearsay_error", "Military Secretary is not military service"),
    11556:  ("hearsay_error", "report treated as fact"),
    83901:  ("hearsay_error", "endorsed products, not Ivanka"),
}

In [29]:
categories = pd.Series([v[0] for v in adjudication.values()])
print("Mistakes adjudicated:", len(categories))
print(categories.value_counts())

Mistakes adjudicated: 42
fever_wrong       24
hearsay_error     11
over_inference     7
Name: count, dtype: int64


In [31]:
fever_wrong_ids = []
for claim_id in adjudication:
    category = adjudication[claim_id][0]
    if category == "fever_wrong":
        fever_wrong_ids.append(claim_id)

df["adj_correct"] = df["correct"] | df["id"].isin(fever_wrong_ids)


print("Raw accuracy:", round(df["correct"].mean() * 100, 1), "%")
print("Adjudicated accuracy:", round(df["adj_correct"].mean() * 100, 1), "%")

print(df.groupby("group")[["correct", "adj_correct"]].mean())

Raw accuracy: 65.6 %
Adjudicated accuracy: 85.2 %
           correct  adj_correct
group                          
NEI-A     0.352941     0.764706
NEI-B     0.500000     0.850000
REFUTES   0.852941     0.882353
SUPPORTS  0.852941     0.911765
